# Smart Room — Occupancy classification

Predict whether the room is occupied (`Occupancy = 1`) or empty (`Occupancy = 0`) from four sensor readings: `Temperature`, `Humidity`, `Light` and `CO2`.

In [ ]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = "Occupancy.csv"      # keep the notebook and the CSV in the same folder
TARGET = "Occupancy"
FEATURES = ["Temperature", "Humidity", "Light", "CO2"]
TEST_SIZE = 0.20
SEED = 42

## 1. Load the data

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print(f"{len(df):,} rows, from {df['date'].min()} to {df['date'].max()}")

## 2. Split: the past for training, the future for testing

The readings are one minute apart, so a random split would put almost identical rows in both sets. Instead, the first 80 % of the timeline is the training set and the last 20 % is the test set.

In [ ]:
X = df[FEATURES]
y = df[TARGET]

split_at = len(df) - round(len(df) * TEST_SIZE)

X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y.iloc[:split_at], y.iloc[split_at:]

print(f"train: {len(X_train):,} rows  |  test: {len(X_test):,} rows")

## 3. Train the classifier

Logistic Regression works best when the features are on a similar scale, so a `StandardScaler` comes first. Both steps live in one `Pipeline`, which means the scaler learns its mean and standard deviation from the training set only.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=SEED)),
])

model.fit(X_train, y_train)

## 4. Predict and evaluate on the test set

In [ ]:
y_pred = model.predict(X_test)      # 0 = empty, 1 = occupied

print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"F1       : {f1_score(y_test, y_pred):.3f}   <- main metric (occupied class)")